# Memory AI Lab — 08 : Évaluation qualitative par LLM

**GPU recommandé** (segmentation) · **Clé API Claude requise**

## Objectif

L'ARI mesure la structure, pas le sens. Ce notebook demande à Claude de juger
si chaque coupure/fusion est **sémantiquement justifiée**.

```
Test set
    ↓ HybridEpisodeSegmenter (Transformer)
    ↓ Identifier TP / FP / FN
    ↓ Échantillonner N cas de chaque type
    ↓ Claude API  →  { justified, confidence, reason }
    ↓ Synthèse qualitative
```

## 3 types de cas analysés

| Type | Définition | Question posée à Claude |
|------|-----------|------------------------|
| **TP** | Coupure correcte (pred ∩ gold) | Pourquoi cette frontière est-elle naturelle ? |
| **FP** | Fausse coupure (pred \ gold) | Cette séparation est-elle injustifiée ? |
| **FN** | Frontière manquée (gold \ pred) | Aurait-on dû couper ici ? |

## Données requises
```
group_anon.txt            (messages anonymisés)
group_gold_test.json
group_embeddings_me5.npy
boundary_detector_tfm.pt  (ou tcn / mlp)
```

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO     = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!pip install anthropic -q
print('✓ OK')

In [ ]:
# ── CELLULE 3 : Drive + Clé API ───────────────────────────────────────────
import os, shutil, getpass
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/memory_ai_data'
LOCAL_DIR = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

for fname in ['group_anon.txt', 'group_gold_test.json',
              'group_embeddings_me5.npy',
              'boundary_detector_tfm.pt',
              'boundary_detector_tcn.pt',
              'boundary_detector.pt']:
    src, dst = f'{DRIVE_DIR}/{fname}', f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'  Copié : {fname} ✓')
    elif os.path.exists(dst):
        print(f'  {fname} déjà en local ✓')
    else:
        print(f'  ⚠️  {fname} absent (ignoré)')

DATA_DIR = LOCAL_DIR

# Clé API Claude — saisie sécurisée (non affichée)
ANTHROPIC_API_KEY = getpass.getpass('Clé API Claude (sk-ant-...) : ')
print('✓ Clé API enregistrée')

In [ ]:
# ── CELLULE 4 : Charger données + pipeline ────────────────────────────────
import numpy as np
import json
import torch
from parsers.whatsapp_parser import parse_whatsapp_chat

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

# Artefacts + embeddings
all_artifacts  = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
all_embeddings = np.load(f'{DATA_DIR}/group_embeddings_me5.npy')

# Gold test
with open(f'{DATA_DIR}/group_gold_test.json') as f:
    gold_data = json.load(f)

gold_episodes = gold_data['episodes']
n_test  = len(gold_data['artifacts'])
n_tune  = len(all_artifacts) - n_test

test_artifacts  = all_artifacts[n_tune:n_tune + n_test]
test_embeddings = all_embeddings[n_tune:n_tune + n_test]

# Labels gold
y_true = [None] * n_test
gold_boundaries = set()
for ep in gold_episodes:
    for idx in range(ep['start_idx'], ep['end_idx'] + 1):
        if idx < n_test:
            y_true[idx] = ep['episode_id']
    if ep['start_idx'] > 0:
        gold_boundaries.add(ep['start_idx'])

print(f'Test set : {n_test} msgs · {len(gold_episodes)} épisodes gold')

# Boundary detector — priorité Transformer > TCN > MLP
DETECTOR = None
for path, cls_name, mod_name in [
    (f'{DATA_DIR}/boundary_detector_tfm.pt', 'TransformerBoundaryDetector', 'boundary_detector_transformer'),
    (f'{DATA_DIR}/boundary_detector_tcn.pt', 'TCNBoundaryDetector',         'boundary_detector_tcn'),
]:
    if os.path.exists(path):
        import importlib
        mod = importlib.import_module(mod_name)
        cls = getattr(mod, cls_name)
        DETECTOR = cls(device=device).load(path)
        print(f'✓ {cls_name} chargé (seuil={DETECTOR.threshold:.3f})')
        break

# Segmentation
from episode_segmenter_hybrid import HybridEpisodeSegmenter
from episode_merger import EpisodeMerger
from episode_resegmenter_fast import EpisodeResegmenterFast

PARAMS = dict(
    attach_threshold       = 0.434,
    ema_alpha              = 0.787,
    time_threshold_minutes = 330,
    boundary_k             = 0.107,
    alpha=0.45, beta=0.25, gamma=0.10, delta=0.20, rho=0.05,
    dormancy_minutes       = 1440,
    hard_break_minutes     = 0,
    active_penalty_hours   = 24.0,
    allow_reactivation     = True,
)
MERGER_PARAMS = dict(min_size=1, merge_sim=0.918, merge_gap_minutes=146.)
FIXED_RESEG   = dict(max_iter=3, window_minutes=480., min_ep_size=2)

seg    = HybridEpisodeSegmenter(detector=DETECTOR, device=device, **PARAMS)
eps    = seg.consolidate(seg.segment(test_artifacts, test_embeddings))
eps    = EpisodeMerger(device=device, **MERGER_PARAMS).merge(eps)
eps    = EpisodeResegmenterFast(**FIXED_RESEG).resegment(eps, test_artifacts, test_embeddings, seg)

# Frontières prédites
pred_boundaries = set()
for ep in eps:
    if ep.artifact_indices:
        start = min(ep.artifact_indices)
        if start > 0:
            pred_boundaries.add(start)

TP = pred_boundaries & gold_boundaries
FP = pred_boundaries - gold_boundaries
FN = gold_boundaries - pred_boundaries

print(f'\nSegmentation : {len(eps)} épisodes prédits / {len(gold_episodes)} gold')
print(f'TP={len(TP)}  FP={len(FP)}  FN={len(FN)}')

In [ ]:
# ── CELLULE 5 : Formater les cas pour Claude ──────────────────────────────
import random
random.seed(42)

N_CASES    = 5     # cas par type (TP, FP, FN) — ajuster selon budget API
CONTEXT_K  = 6     # messages de contexte avant et après la frontière

def format_messages(artifacts, start, end):
    """Formate une fenêtre de messages pour le prompt."""
    lines = []
    for i in range(start, end):
        if 0 <= i < len(artifacts):
            art = artifacts[i]
            ts  = art.timestamp.strftime('%H:%M') if art.timestamp else '??:??'
            author = art.author or 'inconnu'
            lines.append(f'[{ts}] {author}: {art.content}')
    return '\n'.join(lines) if lines else '(vide)'

def build_case(boundary_pos, case_type):
    """Construit un cas à soumettre à Claude."""
    before = format_messages(test_artifacts, boundary_pos - CONTEXT_K, boundary_pos)
    after  = format_messages(test_artifacts, boundary_pos, boundary_pos + CONTEXT_K)

    type_desc = {
        'TP': 'VRAIE FRONTIÈRE (présente dans les deux : prediction et gold)',
        'FP': 'FAUSSE COUPURE (le modèle a créé une frontière — gold dit non)',
        'FN': 'FRONTIÈRE MANQUÉE (gold dit oui — le modèle n\'a pas coupé)',
    }[case_type]

    return {
        'pos':       boundary_pos,
        'type':      case_type,
        'type_desc': type_desc,
        'before':    before,
        'after':     after,
    }

# Échantillonnage aléatoire
cases = []
for label, boundary_set in [('TP', TP), ('FP', FP), ('FN', FN)]:
    sample = random.sample(sorted(boundary_set), min(N_CASES, len(boundary_set)))
    cases += [build_case(pos, label) for pos in sample]

random.shuffle(cases)  # mélanger pour éviter biais d'ordre

print(f'✓ {len(cases)} cas préparés ({N_CASES} TP + {N_CASES} FP + {N_CASES} FN)')
print(f'  Contexte : {CONTEXT_K} messages avant + {CONTEXT_K} après chaque frontière')

In [ ]:
# ── CELLULE 6 : Prompt + appel Claude API ─────────────────────────────────
import anthropic
import json
import time

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """Tu es un expert en analyse de conversations de groupe.
Tu analyses des frontières de segmentation en épisodes thématiques.

Un épisode = un fil de discussion cohérent sur un sujet ou une tâche.
Une frontière = le point où la conversation change de sujet.

Réponds UNIQUEMENT en JSON valide, sans texte autour."""

def build_prompt(case):
    return f"""Analyse cette frontière de conversation.

TYPE : {case['type_desc']}

=== AVANT LA FRONTIÈRE ===
{case['before']}

━━━━━━ FRONTIÈRE ━━━━━━

=== APRÈS LA FRONTIÈRE ===
{case['after']}

Évalue si cette frontière est sémantiquement justifiée.
Réponds en JSON :
{{
  "justified": true/false,
  "confidence": 0.0-1.0,
  "topic_before": "sujet principal avant (1 phrase)",
  "topic_after": "sujet principal après (1 phrase)",
  "reason": "explication en 2-3 phrases"
}}"""

results = []

for i, case in enumerate(cases):
    print(f'\n[{i+1}/{len(cases)}] pos={case["pos"]} type={case["type"]} ...', end=' ', flush=True)

    try:
        response = client.messages.create(
            model     = 'claude-sonnet-4-6',
            max_tokens = 512,
            system    = SYSTEM_PROMPT,
            messages  = [{'role': 'user', 'content': build_prompt(case)}],
        )
        raw = response.content[0].text.strip()

        # Parser le JSON
        if raw.startswith('```'):
            raw = raw.split('```')[1].lstrip('json').strip()
        analysis = json.loads(raw)

        result = {**case, 'analysis': analysis, 'error': None}
        print(f'justified={analysis["justified"]}  conf={analysis["confidence"]:.2f} ✓')

    except Exception as e:
        result = {**case, 'analysis': None, 'error': str(e)}
        print(f'ERREUR: {e}')

    results.append(result)
    time.sleep(0.5)   # politesse API

print(f'\n✓ {sum(1 for r in results if r["error"] is None)}/{len(results)} appels réussis')

In [ ]:
# ── CELLULE 7 : Synthèse par type ─────────────────────────────────────────
import numpy as np

print('=' * 65)
print('  SYNTHÈSE — APPRÉCIATION QUALITATIVE PAR CLAUDE')
print('=' * 65)

for case_type in ['TP', 'FP', 'FN']:
    subset = [r for r in results if r['type'] == case_type and r['analysis']]
    if not subset:
        continue

    justified  = [r['analysis']['justified']  for r in subset]
    confidence = [r['analysis']['confidence'] for r in subset]

    pct_justified = 100 * sum(justified) / len(justified)
    avg_conf      = np.mean(confidence)

    label = {
        'TP': 'Vraies frontières  (TP)',
        'FP': 'Fausses coupures   (FP)',
        'FN': 'Frontières manquées(FN)',
    }[case_type]

    print(f'\n── {label} ──')
    print(f'   Justifiées selon Claude : {pct_justified:.0f}%  ({sum(justified)}/{len(justified)})')
    print(f'   Confiance moyenne       : {avg_conf:.2f}')

    for r in subset:
        a = r['analysis']
        icon = '✓' if a['justified'] else '✗'
        print(f'   {icon} pos={r["pos"]:4d}  conf={a["confidence"]:.2f}')
        print(f'       Avant : {a["topic_before"]}')
        print(f'       Après : {a["topic_after"]}')
        print(f'       → {a["reason"]}')

print('\n' + '=' * 65)
print('  INTERPRÉTATION')
print('=' * 65)

tp_ok = [r for r in results if r['type']=='TP' and r['analysis'] and r['analysis']['justified']]
fp_ok = [r for r in results if r['type']=='FP' and r['analysis'] and not r['analysis']['justified']]
fn_ok = [r for r in results if r['type']=='FN' and r['analysis'] and r['analysis']['justified']]

n_tp = len([r for r in results if r['type']=='TP' and r['analysis']])
n_fp = len([r for r in results if r['type']=='FP' and r['analysis']])
n_fn = len([r for r in results if r['type']=='FN' and r['analysis']])

print(f'\n  TP justifiées     : {len(tp_ok)}/{n_tp} — le modèle coupe aux bons endroits')
print(f'  FP non justifiées : {len(fp_ok)}/{n_fp} — fausses coupures confirmées par Claude')
print(f'  FN justifiées     : {len(fn_ok)}/{n_fn} — frontières manquées confirmées par Claude')

fp_justified = [r for r in results if r['type']=='FP' and r['analysis'] and r['analysis']['justified']]
if fp_justified:
    print(f'\n  ⚡ {len(fp_justified)} FP jugées JUSTIFIÉES par Claude')
    print('    → Ces coupures sont sémantiquement valides malgré l\'absence dans le gold')
    print('    → Possible sous-annotation dans le gold standard')

In [ ]:
# ── CELLULE 8 : Vue détaillée — cas les plus intéressants ─────────────────
# FP jugées justifiées par Claude = potentielle sous-annotation du gold
# FN jugées justifiées = frontières que le modèle aurait dû trouver

interesting = [
    r for r in results
    if r['analysis'] and (
        # FP justifiée = coupure défendable malgré absence dans le gold
        (r['type'] == 'FP' and r['analysis']['justified']) or
        # FN justifiée = frontière manquée confirmée
        (r['type'] == 'FN' and r['analysis']['justified'] and r['analysis']['confidence'] > 0.7)
    )
]

if not interesting:
    print('Aucun cas particulièrement intéressant — le modèle et le gold sont alignés.')
else:
    print(f'{len(interesting)} cas notables :\n')
    for r in interesting:
        a = r['analysis']
        print(f'┌─ {r["type"]}  pos={r["pos"]}  conf={a["confidence"]:.2f}')
        print(f'│  Avant : {a["topic_before"]}')
        print(f'│  Après : {a["topic_after"]}')
        print(f'│  → {a["reason"]}')
        print(f'└───────────────────────────────────────────────────────────')

In [ ]:
# ── CELLULE 9 : Sauvegarde des résultats ──────────────────────────────────
import json, shutil
from datetime import datetime

OUTPUT = {
    'date':    datetime.now().isoformat(),
    'model':   'claude-sonnet-4-6',
    'n_cases': len(results),
    'results': [
        {
            'pos':      r['pos'],
            'type':     r['type'],
            'analysis': r['analysis'],
        }
        for r in results if r['analysis']
    ]
}

LOCAL_PATH = f'{DATA_DIR}/qualitative_eval.json'
DRIVE_PATH = f'{DRIVE_DIR}/qualitative_eval.json'

with open(LOCAL_PATH, 'w', encoding='utf-8') as f:
    json.dump(OUTPUT, f, ensure_ascii=False, indent=2)

shutil.copy2(LOCAL_PATH, DRIVE_PATH)
print(f'✓ Résultats sauvegardés → Drive : qualitative_eval.json')